In [ ]:
%%capture
#!pip install vllm triton
!pip install vllm transformers==4.49 trl peft

#  Language Models

## Playground

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch

model_path="ibm-granite/granite-3.2-8b-instruct"
device="auto"
model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.bfloat16,
    )
tokenizer = AutoTokenizer.from_pretrained(
        model_path
)

model

In [ ]:

conv = [{"role": "user", "content":"You have 10 liters of a 30% acid solution. How many liters of a 70% acid solution must be added to achieve a 50% acid mixture?"}]

input_ids = tokenizer.apply_chat_template(conv, return_tensors="pt", thinking=True, return_dict=True, add_generation_prompt=True).to(model.device)

set_seed(42)
output = model.generate(
    **input_ids,
    max_new_tokens=8192,
)

In [ ]:
from IPython.display import Markdown
prediction = tokenizer.decode(output[0, input_ids["input_ids"].shape[1]:], skip_special_tokens=True)
Markdown(prediction)

In [ ]:
conv = [
    {"role": "system", "content":"柔情女医生口吻"},
    {"role": "user", "content":"怎么样做运动?"},
]

input_ids = tokenizer.apply_chat_template(
    conv, return_tensors="pt", thinking=True, return_dict=True, add_generation_prompt=True).to(model.device)

set_seed(42)
output = model.generate(
    **input_ids,
    max_new_tokens=8192,
)

In [ ]:
from IPython.display import Markdown
prediction = tokenizer.decode(output[0, input_ids["input_ids"].shape[1]:], skip_special_tokens=True)
Markdown(prediction)

## Inference

In [ ]:
from vllm import LLM, SamplingParams

model_path = "ibm-granite/granite-3.2-8b-instruct"
model = LLM(
    model=model_path,
    gpu_memory_utilization=0.8,tensor_parallel_size=2,
    distributed_executor_backend='ray',
    dtype="float16"
)

In [ ]:
sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=64,
)

prompt = "What is the highest scoring model on ChartQA and what is its score?"

outputs = model.generate(prompt, sampling_params=sampling_params)
print(f"Generated text: {outputs[0].outputs[0].text}")


# Vision Models

## Playground

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

#device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = "ibm-granite/granite-vision-3.2-2b"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForVision2Seq.from_pretrained(model_path, device_map="auto")
model

In [ ]:
from huggingface_hub import hf_hub_download
# prepare image and text prompt, using the appropriate prompt template

img_path = hf_hub_download(repo_id=model_path, filename='example.png')

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What is the highest scoring model on ChartQA and what is its score?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)


# autoregressively complete prompt
output = model.generate(**inputs, max_new_tokens=100)
print(processor.decode(output[0], skip_special_tokens=True))


In [ ]:
# prepare image and text prompt, using the appropriate prompt template

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "/kaggle/input/diffusion/HXRlOIUV2iP4FEgPKSVYH.png"},
            {"type": "text", "text": "什么?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)


# autoregressively complete prompt
output = model.generate(**inputs, max_new_tokens=100)
print(processor.decode(output[0], skip_special_tokens=True))

## Inference

In [ ]:
from vllm import LLM, SamplingParams
from vllm.assets.image import ImageAsset
from PIL import Image

model_path = "ibm-granite/granite-vision-3.2-2b"

model = LLM(
    model=model_path,
    gpu_memory_utilization=0.8,tensor_parallel_size=2,
    distributed_executor_backend='ray',
    limit_mm_per_prompt={"image": 1},
    dtype="float16"
)

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=64,
)

In [ ]:
# Define the question we want to answer and format the prompt
from huggingface_hub import hf_hub_download

image_token = "<image>"
system_prompt = "<|system|>\nA chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.\n"

question = "What is the highest scoring model on ChartQA and what is its score?"
prompt = f"{system_prompt}<|user|>\n{image_token}\n{question}\n<|assistant|>\n"
img_path = hf_hub_download(repo_id=model_path, filename='example.png')
image = Image.open(img_path).convert("RGB")
print(image)

# Build the inputs to vLLM; the image is passed as `multi_modal_data`.
inputs = {
    "prompt": prompt,
    "multi_modal_data": {
        "image": image,
    }
}

outputs = model.generate(inputs, sampling_params=sampling_params)
print(f"Generated text: {outputs[0].outputs[0].text}")